# Aggregation in DeepLog
Aggregation is the fundamental mechanism in DeepLog that allows formulas to range over multiple assignments and combine their contributions into a single value.

An aggregation specifies:
- how to aggregate (sum, max, product, …),
- what to aggregate over (variables),
- and what formula to evaluate for each assignment.  

Syntactically, aggregation is written as:
```
operator(Variables): <subformula>
```

For example, `sum(X): <subformula>` sums the results of the subformula over all values in the domain of `X`.

## Operational Semantics of Aggregation
At compile time, every aggregation node in a formula is dispatched to an `AggregationBuilder`. The builder decides how to implement the aggregation. For operators like `sum`, the builder creates an `AggregationModule`:
```
AggregationModule(aggregated_module, variables, domains, name, op)
```

where `variables` is a list of bound variables, `domains` is a list of corresponding domain tensors, `name` identifies the aggregation, and `op` is the reduction operation.

Other builders (e.g. `expectation`) may compile the aggregation differently, for instance by reinterpreting the child circuit in a different semiring.

At runtime, `AggregationModule.forward()` implements the following steps:
- Broadcast the input across all values in the domain of the aggregated variable.
- Insert each domain value into the correct column of the symbolic assignment.
- Evaluate the aggregated submodule on all expanded assignments.
- Reshape the result into: `(batch_size, domain_size, ...)`
- Reduce along the domain dimension using the selected operator.

This implements the algebraic computation exactly as a vectorized tensor operation.

## Model Counting via Aggregation

We start with the simplest example: plain model counting.  
Consider two Boolean variables: `Burglary` and `Earthquake`

We want to count how many Boolean assignments satisfy:
```
Burglary = true OR Earthquake = true
```

In [ ]:
# Formula
model_count_text = """
sum(Burglary, Earthquake):
    =(Burglary,true)_boolean or =(Earthquake,true)_boolean
"""

## Compilation and Evaluation


In [ ]:
import torch

from deeplog import parse_formula_to_module


model_count_module = parse_formula_to_module(model_count_text)
model_count_module

In [ ]:
model_count = model_count_module()
print("Model count:", model_count)


### What Happens Internally
For the aggregation sum (Burglary, Earthquake):
- The domains of both variables have two elements: {false, true}.
- The input batch is repeated four times.
- The columns corresponding to  and Earthquake are filled with `[[0, 0], [0, 1], [1,0], [1, 1]]`.
- The inner module is evaluated on all assignments.
- The results are summed.

This constructs the full Cartesian product of assignments purely via tensor broadcasting.

## Aggregation Over Non-Boolean Domains

Aggregation works over any declared finite domain, not only Booleans.

Example: count how many digits in {0,…,9} are even.

In [ ]:
# Formula
even_formula = """
sum(Digit):
    even(Digit)_boolean
"""

In [ ]:
# Custom predicate:

import torch

from deeplog import Predicate, Symbol


class EvenPredicate(Predicate):
    functor = "even"
    arity = 1
    structure = "boolean"

    def _resolve_argument(self, symbol: Symbol, index: int):
        try:
            return int(symbol[0])
        except (ValueError, TypeError, IndexError):
            return symbol

    def forward_predicate(self, digits: torch.Tensor) -> torch.Tensor:
        return (digits.remainder(2) == 0).to(digits.dtype)

In [ ]:
import torch

from deeplog import DeepLogModuleFactory, Symbol, parse_formula_to_module


digit = ("Digit",)
digit_domain = torch.arange(10)

variables: dict[Symbol, torch.Tensor] = {digit: digit_domain}

factory = DeepLogModuleFactory(
    variables=variables,
    atom_builders={("even", 1, "boolean"): EvenPredicate},
)

even_module = parse_formula_to_module(
    even_formula,
    factory=factory,
)

print("Number of even digits in the given range:", even_module())

## Aggregation Operators

Aggregation operators are registered as `AggregationBuilder` callables in `DeepLogModuleFactory.aggregators`. Each builder receives the child node, bound variables, parameters, and domains, and returns an `InternalNode` representing the aggregation.

In [ ]:
from functools import partial

from deeplog import AggregationModule

# The default "sum" aggregation builder is registered as:
sum_builder = partial(AggregationModule, name="sum", op=lambda x: x.sum(dim=1))

# Custom aggregation builders can be passed to DeepLogModuleFactory:
# factory = DeepLogModuleFactory(aggregators={"my_op": my_builder})

Currently supported default aggregation builders:

- **sum** — Reduces along the domain dimension using `x.sum(dim=1)`. Used for counting, total mass, weighted sums, etc.
- **expectation** — Compiles a boolean circuit child in the probability semiring. Used for computing expected values over probabilistic models.

## Aggregation and Free Variables

A variable is bound if it appears in an aggregation operator.
A variable is free if it does not.

Only free variables become inputs to the compiled module.

In [ ]:
# Example:

formula = """
sum(Digit):
    greater_than(Digit, X)_boolean
"""

`Digit` is bound by aggregation. `X` is free.

The compiled module expects `X` as part of its input tensor.

## Aggregation and Differentiability

Because aggregation is implemented using:
- tensor broadcasting,
- pure torch operations,
- and reduction operators,

it is fully differentiable.